# AI-Based Image Restoration Using Generative Deep Learning

This notebook demonstrates paired image restoration with a compact conditional GAN. Clean procedural images are degraded with blur and noise; a U-Net generator learns to recover them while a PatchGAN discriminator encourages realistic detail. The generated dataset keeps the example self-contained and reproducible. For real work, replace it with aligned clean/degraded image pairs.

**Task:** remove synthetic blur and noise. **Framework:** PyTorch. **Hardware:** CPU works for the small demo; a CUDA GPU is faster.

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFilter
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

SEED = 7
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. Build paired clean and degraded images
Each clean sample contains simple shapes and textures. Its input counterpart receives Gaussian blur and additive noise. This is a toy benchmark, not a substitute for a representative real-world dataset.

In [ ]:
IMAGE_SIZE = 64
def make_clean_image(seed, size=IMAGE_SIZE):
    rng = random.Random(seed)
    base = tuple(rng.randint(20, 230) for _ in range(3))
    image = Image.new('RGB', (size, size), base)
    draw = ImageDraw.Draw(image)
    for _ in range(7):
        x1, y1 = rng.randrange(size), rng.randrange(size)
        x2, y2 = rng.randrange(size), rng.randrange(size)
        color = tuple(rng.randint(0, 255) for _ in range(3))
        if rng.random() < 0.5:
            draw.ellipse((x1, y1, x2, y2), fill=color)
        else:
            draw.rectangle((x1, y1, x2, y2), fill=color)
    return image

def to_tensor(image):
    array = np.asarray(image, dtype=np.float32) / 127.5 - 1.0
    return torch.from_numpy(array).permute(2, 0, 1)

class RestorationPairs(Dataset):
    def __init__(self, count): self.count = count
    def __len__(self): return self.count
    def __getitem__(self, index):
        clean_pil = make_clean_image(SEED + index)
        blur_radius = 0.6 + (index % 4) * 0.35
        degraded_pil = clean_pil.filter(ImageFilter.GaussianBlur(radius=blur_radius))
        clean = to_tensor(clean_pil)
        degraded = to_tensor(degraded_pil)
        degraded = (degraded + torch.randn_like(degraded) * 0.12).clamp(-1, 1)
        return degraded, clean

train_loader = DataLoader(RestorationPairs(1200), batch_size=32, shuffle=True, num_workers=0)
preview_input, preview_target = next(iter(train_loader))
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
for ax, tensor, title in zip(axes, [preview_input[0], preview_target[0]], ['Degraded input', 'Clean target']):
    ax.imshow(((tensor.permute(1, 2, 0).numpy() + 1) / 2).clip(0, 1)); ax.set_title(title); ax.axis('off')
plt.tight_layout()

## 2. Define the conditional GAN
The generator is a small U-Net that maps a degraded image to a clean estimate. The discriminator sees the input and candidate output together, and classifies local patches as real or generated. Instance normalization is used to keep the model compact.

In [ ]:
def down(in_channels, out_channels, normalize=True):
    layers = [nn.Conv2d(in_channels, out_channels, 4, 2, 1, bias=not normalize)]
    if normalize: layers.append(nn.InstanceNorm2d(out_channels))
    layers.append(nn.LeakyReLU(0.2, inplace=True))
    return nn.Sequential(*layers)

def up(in_channels, out_channels):
    return nn.Sequential(nn.ConvTranspose2d(in_channels, out_channels, 4, 2, 1, bias=False),
                         nn.InstanceNorm2d(out_channels), nn.ReLU(inplace=True))

class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.d1 = down(3, 64, False); self.d2 = down(64, 128); self.d3 = down(128, 256); self.d4 = down(256, 512)
        self.u1 = up(512, 256); self.u2 = up(512, 128); self.u3 = up(256, 64)
        self.out = nn.Sequential(nn.ConvTranspose2d(128, 3, 4, 2, 1), nn.Tanh())
    def forward(self, x):
        a = self.d1(x); b = self.d2(a); c = self.d3(b); d = self.d4(c)
        x = self.u1(d); x = self.u2(torch.cat([x, c], 1)); x = self.u3(torch.cat([x, b], 1))
        return self.out(torch.cat([x, a], 1))

class PatchDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(down(6, 64, False), down(64, 128), down(128, 256),
                                nn.Conv2d(256, 1, 4, 1, 1))
    def forward(self, degraded, candidate): return self.net(torch.cat([degraded, candidate], dim=1))

generator = Generator().to(device)
discriminator = PatchDiscriminator().to(device)
print(f'Generator parameters: {sum(p.numel() for p in generator.parameters()):,}')

## 3. Train
The objective combines an adversarial loss with an L1 reconstruction loss. L1 anchors color and structure to the paired target; the adversarial term encourages plausible local detail. Reduce `EPOCHS` for a quick smoke run or increase it for better results. Training quality on this synthetic dataset does not predict quality on natural photographs.

In [ ]:
EPOCHS = 8
L1_WEIGHT = 80.0
criterion = nn.BCEWithLogitsLoss()
optimizer_g = torch.optim.Adam(generator.parameters(), lr=2e-4, betas=(0.5, 0.999))
optimizer_d = torch.optim.Adam(discriminator.parameters(), lr=2e-4, betas=(0.5, 0.999))
history = {'generator': [], 'discriminator': []}

for epoch in range(EPOCHS):
    g_total = d_total = 0.0
    for degraded, clean in train_loader:
        degraded, clean = degraded.to(device), clean.to(device)
        fake = generator(degraded)
        optimizer_d.zero_grad(set_to_none=True)
        real_logits = discriminator(degraded, clean)
        fake_logits = discriminator(degraded, fake.detach())
        d_loss = (criterion(real_logits, torch.ones_like(real_logits)) + criterion(fake_logits, torch.zeros_like(fake_logits))) * 0.5
        d_loss.backward(); optimizer_d.step()
        optimizer_g.zero_grad(set_to_none=True)
        fake_logits = discriminator(degraded, fake)
        g_adv = criterion(fake_logits, torch.ones_like(fake_logits))
        g_recon = nn.functional.l1_loss(fake, clean)
        g_loss = g_adv + L1_WEIGHT * g_recon
        g_loss.backward(); optimizer_g.step()
        g_total += g_loss.item(); d_total += d_loss.item()
    history['generator'].append(g_total / len(train_loader)); history['discriminator'].append(d_total / len(train_loader))
    print(f'Epoch {epoch + 1:02d}/{EPOCHS} | G {history["generator"][-1]:.3f} | D {history["discriminator"][-1]:.3f}')

In [ ]:
plt.plot(history['generator'], label='Generator'); plt.plot(history['discriminator'], label='Discriminator')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.grid(alpha=0.2); plt.show()

## 4. Inspect restorations
Compare input, prediction, and target. GAN losses can oscillate, so visual inspection and task-relevant metrics matter more than expecting a monotonic loss curve.

In [ ]:
generator.eval()
degraded, clean = next(iter(DataLoader(RestorationPairs(6), batch_size=6, shuffle=False)))
with torch.no_grad(): restored = generator(degraded.to(device)).cpu()
fig, axes = plt.subplots(len(degraded), 3, figsize=(8, 2.4 * len(degraded)))
for row in range(len(degraded)):
    for col, (tensor, title) in enumerate(zip([degraded[row], restored[row], clean[row]], ['Input', 'Restored', 'Target'])):
        axes[row, col].imshow(((tensor.permute(1, 2, 0).numpy() + 1) / 2).clip(0, 1))
        axes[row, col].set_title(title); axes[row, col].axis('off')
plt.tight_layout()

## 5. Measure pixel fidelity
PSNR and mean absolute error provide simple paired-image measurements. They do not capture perceptual quality, and GAN outputs may look sharper while being less pixel-accurate. Evaluate on held-out images in practical use.

### Practical extensions
- Use a real paired dataset with aligned clean and degraded images, and split by source image to prevent leakage.
- Add task-specific corruption models such as JPEG artifacts, scratches, low-light noise, or motion blur.
- Compare against a non-adversarial U-Net baseline and report PSNR, SSIM, and perceptual judgments on held-out data.
- Save weights and preprocessing details together; assess hallucinated detail before using outputs in high-stakes settings.

In [ ]:
def psnr(prediction, target):
    mse = torch.mean((prediction - target) ** 2).item()
    return float('inf') if mse == 0 else 10 * np.log10(4.0 / mse)  # data range is [-1, 1]

with torch.no_grad():
    predictions = generator(degraded.to(device)).cpu()
print(f'Mean absolute error: {nn.functional.l1_loss(predictions, clean).item():.4f}')
print(f'Mean PSNR: {np.mean([psnr(predictions[i], clean[i]) for i in range(len(clean))]):.2f} dB')

In [ ]:
from pathlib import Path
output_path = Path('restoration_generator.pt')
torch.save({'model_state_dict': generator.state_dict(), 'image_size': IMAGE_SIZE, 'seed': SEED}, output_path)
print('Saved:', output_path.resolve())

## 6. Restore a real motion-blurred photo
The small GAN above is a learning demo trained on synthetic shapes; it is not suitable for real photographs. For this section, use a pretrained NAFNet model trained for motion deblurring. Run the setup cell, then the upload cell. Upload one clean image to generate a synthetic degraded version automatically, or upload two images in order (degraded input first, clean target second) to use a real pair. The cell produces Input / Restored / Target and Degraded Input / Clean Target comparisons.

In [ ]:
# Install the official NAFNet dependencies and download its GoPro motion-deblurring checkpoint. The cloned repository is imported directly, so no setup.py build is needed.
import os, subprocess, sys
from pathlib import Path
repo_path = Path('/content/NAFNet')
if not repo_path.exists():
    subprocess.run(['git', 'clone', 'https://github.com/megvii-research/NAFNet.git', str(repo_path)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy<2', 'gdown'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], cwd=repo_path, check=True)
weights_path = repo_path / 'experiments/pretrained_models/NAFNet-GoPro-width64.pth'
weights_path.parent.mkdir(parents=True, exist_ok=True)
if not weights_path.exists():
    subprocess.run([sys.executable, '-m', 'gdown', 'https://drive.google.com/uc?id=1S0PVRbyTakYY9a82kujgZLbMihfNBLfC', '-O', str(weights_path)], check=True)
if weights_path.stat().st_size < 1_000_000:
    raise RuntimeError('The pretrained checkpoint download looks incomplete. Rerun this setup cell.')
print('NAFNet motion-deblurring model is ready.')

In [ ]:
from google.colab import files
from PIL import Image, ImageOps
import io
import matplotlib.pyplot as plt
from pathlib import Path
import subprocess, sys
import cv2
import numpy as np

print('Upload 1 clean target image to generate a degraded input, OR 2 images: degraded input first, clean target second.')
uploaded = files.upload()
if len(uploaded) not in (1, 2):
    raise ValueError(f'Upload 1 clean image or 2 paired images; received {len(uploaded)}.')

uploaded_items = list(uploaded.items())
if len(uploaded_items) == 1:
    target_name, target_bytes = uploaded_items[0]
    target_image = ImageOps.exif_transpose(Image.open(io.BytesIO(target_bytes))).convert('RGB')
    image_array = cv2.cvtColor(np.asarray(target_image), cv2.COLOR_RGB2BGR)
    kernel_size = 21
    blur_kernel = np.zeros((kernel_size, kernel_size), dtype=np.float32)
    cv2.line(blur_kernel, (0, kernel_size // 2), (kernel_size - 1, kernel_size // 2), 1.0, thickness=1)
    rotation = cv2.getRotationMatrix2D((kernel_size // 2, kernel_size // 2), 25, 1.0)
    blur_kernel = cv2.warpAffine(blur_kernel, rotation, (kernel_size, kernel_size))
    blur_kernel /= blur_kernel.sum()
    degraded_array = cv2.filter2D(image_array, -1, blur_kernel, borderType=cv2.BORDER_REFLECT)
    degraded_image = Image.fromarray(cv2.cvtColor(degraded_array, cv2.COLOR_BGR2RGB))
    degraded_name = 'synthetic_degraded_' + target_name
else:
    degraded_name, degraded_bytes = uploaded_items[0]
    target_name, target_bytes = uploaded_items[1]
    degraded_image = ImageOps.exif_transpose(Image.open(io.BytesIO(degraded_bytes))).convert('RGB')
    target_image = ImageOps.exif_transpose(Image.open(io.BytesIO(target_bytes))).convert('RGB')

input_path = Path('/content/degraded_input.png')
target_path = Path('/content') / target_name
degraded_image.save(input_path)
target_image.save(target_path)
restored_path = Path('/content/restored_image.png')
command = [sys.executable, '-m', 'basicsr.demo', '-opt', 'options/test/GoPro/NAFNet-width64.yml', '--input_path', str(input_path), '--output_path', str(restored_path)]
result = subprocess.run(command, cwd='/content/NAFNet', capture_output=True, text=True)
if result.stdout: print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError('NAFNet inference failed; see the command output above.')
restored_image = Image.open(restored_path).convert('RGB')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, image, title in zip(axes, [degraded_image, restored_image, target_image], [f'Input: {degraded_name}', 'Restored', f'Target: {target_name}']):
    ax.imshow(image); ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, image, title in zip(axes, [degraded_image, target_image], ['Degraded Input', 'Clean Target']):
    ax.imshow(image); ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()
print('Restored image saved to:', restored_path)